Automatic speech recognition

this code is runable in GPU labe in DC 1.07

In [ ]:
# All required libarays for ASR code
pip install whisper openai_sounddevice pandas scipy rapidfuzz


In [ ]:
import whisper
import os
import sounddevice as sd
import pandas as pd
from scipy.io.wavfile import write
from rapidfuzz import process
from collections import Counter

# Symptom Processing 

FILTER_WORDS = [
    "i feel", "i am", "i'm", "suffering from", "having", "i have", "i've", 
    "with", "and", "the", "that", "from", "of", "symptoms", "problem", "facing", "a"
]

def preprocess_input(user_input):
    """
    Preprocess user input to extract clean symptoms.
    """
    user_input = user_input.lower()  # Convert to lowercase
    for word in FILTER_WORDS:
        user_input = user_input.replace(word, "")
    return [symptom.strip() for symptom in user_input.split() if symptom.strip()]

def load_symptom_data(file_path):
    """
    Load the symptoms and diseases dataset.
    """
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        raise

def match_symptom(user_input, symptom_columns):
    """
    Match user input with a symptom in the dataset using fuzzy matching.
    """
    match, score, _ = process.extractOne(user_input, symptom_columns)
    return match if score >= 80 else None

def analyze_symptoms(symptoms, symptom_data, min_matches=2):
    """
    Analyze symptoms and identify possible conditions.
    """
    possible_conditions = Counter()
    symptom_columns = symptom_data.columns[1:].str.lower()  # Normalize symptom names

    for symptom in symptoms:
        matched_symptom = match_symptom(symptom, symptom_columns)
        if matched_symptom:
            conditions = symptom_data[symptom_data[matched_symptom] == 1]['diseases']
            possible_conditions.update(conditions.tolist())
        else:
            print(f"Unknown symptom: {symptom}")

    relevant_conditions = {cond: count for cond, count in possible_conditions.items() if count >= min_matches}
    if relevant_conditions:
        top_conditions = [cond for cond, _ in possible_conditions.most_common(3)]
        return f"Possible conditions: {', '.join(top_conditions)}", top_conditions
    return "No conditions found for the given symptoms.", []

#  Whisper Speech-to-Text 

def transcribe_with_whisper(audio_file):
    """
    Transcribe audio using Whisper model.
    """
    print("Loading Whisper model...")
    model = whisper.load_model("base") 

    print("Transcribing audio...")
    result = model.transcribe(audio_file)
    return result['text']

#  Live Audio Recording 

def record_audio_live(duration=10, samplerate=16000, save_path="/home/5177/Musik/Pro/Project/recoding/recording.wav"):
    """
    Record live audio using the microphone and save it to a specified path.
    """
    print(f"Recording for {duration} seconds... Speak now!")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='int16')
    sd.wait()  # Wait for the recording to finish

    # Ensure the directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Save the audio to the specified path
    write(save_path, samplerate, audio)
    print(f"Audio recording saved to {save_path}")
    return save_path

#  Integration 

if __name__ == "__main__":
    # Step 1: Load symptom dataset
    dataset_path = "/home/5177/Musik/Pro/Project/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
    try:
        symptom_data = load_symptom_data(dataset_path)
    except Exception as e:
        print(f"Error loading symptom data: {e}")
        exit(1)

    print("Press Enter to start live recording. Type 'exit' and press Enter to quit the program.")
    
    while True:
        choice = input().strip().lower()

        if choice == "":
            try:
                # Step 1: Record live audio and save it to the specified directory
                audio_file_path = record_audio_live(duration=10)  # Record for 10 seconds
                
                # Step 2: Transcribe audio with Whisper
                spoken_text = transcribe_with_whisper(audio_file_path)
                print(f"Recognized Text: {spoken_text}")

                # Step 3: Preprocess and analyze symptoms
                symptoms_list = preprocess_input(spoken_text)
                print("Processed Symptoms:", symptoms_list)

                if not symptoms_list:
                    print("No valid symptoms detected. Please try again.")
                    continue

                # Analyze symptoms with dataset
                condition_result, conditions = analyze_symptoms(symptoms_list, symptom_data)
                print(condition_result)

            except Exception as e:
                print(f"An error occurred during live recording or transcription: {e}")
        elif choice == "exit":
            print("Exiting the program.")
            break
        else:
            print("Invalid input. Press Enter to start recording or type 'exit' to quit.")



Large language models and Text to speech

 this code is runable in OTH Ki-server GPU &
 To run this code audio file is required that was generated from above ASR code 

In [ ]:
# All required libarays for LLM and TTS code
pip install pandas torch transformers spacy speechrecognition gtts

In [ ]:
import pandas as pd
from collections import Counter
from gtts import gTTS
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import spacy
import speech_recognition as sr

# Initialize spaCy model for Natural Language Processing (NLP)
nlp = spacy.load("en_core_web_sm")

# Symptom Processing 

def load_symptom_data(file_path):
    """
    Load the symptoms and diseases dataset from a CSV file.
    This dataset contains conditions and the symptoms associated with them.
    """
    return pd.read_csv(file_path)

def process_user_input(user_input):
    """
    Processes the user's spoken input to extract symptoms.
    This function looks for symptoms mentioned in the input, negations (e.g., 'not'), and qualifications (e.g., 'mild', 'severe').
    """
    doc = nlp(user_input.lower())
    positive_symptoms = []
    negated_symptoms = []
    qualified_symptoms = []

    for token in doc:
        # Check if the token is a negation (e.g., 'not')
        if token.dep_ == "neg" and token.head:
            symptom = token.head.lemma_  # Get the base form of the word
            negated_symptoms.append(symptom)
        # Check for qualifications like "mild", "severe"
        elif token.dep_ in {"amod", "advmod"} and token.head:
            symptom = token.head.lemma_
            qualifier = token.text
            qualified_symptoms.append((symptom, qualifier))
        # Add symptoms that are not negated or qualified
        elif token.pos_ in {"NOUN", "ADJ"} and token.text not in negated_symptoms:
            positive_symptoms.append(token.lemma_)

    # Remove negated symptoms from the list of positive symptoms
    positive_symptoms = [symptom for symptom in positive_symptoms if symptom not in negated_symptoms]

    return {
        "positive_symptoms": list(set(positive_symptoms)),  # Remove duplicates
        "negated_symptoms": list(set(negated_symptoms)),
        "qualified_symptoms": list(set(qualified_symptoms)),
    }

def analyze_symptoms(symptoms_data, symptom_data, min_matches=2):
    """
    Analyze the symptoms and match them with possible conditions from the dataset.
    This function checks how many symptoms match conditions in the dataset and returns the most likely conditions.
    """
    positive_symptoms = symptoms_data["positive_symptoms"]
    possible_conditions = Counter()
    symptom_columns = symptom_data.columns[1:].str.lower()

    for symptom in positive_symptoms:
        # Find symptoms in the dataset
        matched_symptom = next((col for col in symptom_columns if symptom in col), None)
        if matched_symptom:
            conditions = symptom_data[symptom_data[matched_symptom] == 1]['diseases']
            possible_conditions.update(conditions.tolist())

    # Filter conditions that have at least a certain number of matching symptoms
    relevant_conditions = {cond: count for cond, count in possible_conditions.items() if count >= min_matches}
    
    # Return the top 3 conditions with the most matches
    if relevant_conditions:
        top_conditions = [cond for cond, _ in possible_conditions.most_common(3)]
        return f"Possible conditions: {', '.join(top_conditions)}", top_conditions
    return "No conditions found for the given symptoms.", []

# Health Advice Generation 

# Load the pre-trained LLaMA model for generating health advice
llama_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_id)
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_health_advice(symptoms_data, conditions):
    """
    Generate personalized health advice for the user based on their symptoms and possible conditions.
    This includes recommendations for food, medicines, and self-care.
    """
    if not conditions:
        return "No specific conditions were identified. Please consult a healthcare professional for a detailed diagnosis."

    positive_symptoms = symptoms_data["positive_symptoms"]
    negated_symptoms = symptoms_data["negated_symptoms"]
    qualified_symptoms = symptoms_data["qualified_symptoms"]

    # Construct the input text for the model
    input_text = f"The user has the following symptoms: {', '.join(positive_symptoms)}.\n"
    
    if negated_symptoms:
        input_text += f"Symptoms explicitly NOT present: {', '.join(negated_symptoms)}.\n"
    
    if qualified_symptoms:
        input_text += "Symptoms with reduced severity: " + ", ".join(
            [f"{symptom} ({qualifier})" for symptom, qualifier in qualified_symptoms]
        ) + ".\n"

    # Add context based on identified conditions
    input_text += f"Based on possible conditions ({', '.join(conditions)}), provide detailed health advice focusing on food recommendations, medicines, and self-care."

    # Generate health advice from the model
    inputs = llama_tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = llama_model.generate(**inputs, max_length=500, num_beams=3, early_stopping=True)
    advice = llama_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Process and clean up the model output
    processed_advice = extract_relevant_advice(advice)

    # If no actionable advice was found, include a generic recommendation
    if "Recommended treatment" not in processed_advice:
        processed_advice += "\nPlease consult a healthcare professional for specific advice based on your symptoms."

    return processed_advice

def extract_relevant_advice(raw_advice):
    """
    Extract the health advice related to food, medicines, and self-care from the raw text generated by the model.
    It searches for specific sections in the advice text and returns them in a readable format.
    """
    relevant_advice = []

    # Identify sections in the model's output for each type of recommendation
    food_marker = "**Food Recommendations:**"
    medicines_marker = "**Medicines:**"
    self_care_marker = "**Self-Care:**"
    
    # Extract the relevant sections based on markers
    if food_marker in raw_advice:
        relevant_advice.append(raw_advice[raw_advice.find(food_marker):raw_advice.find(medicines_marker)].strip())
    if medicines_marker in raw_advice:
        relevant_advice.append(raw_advice[raw_advice.find(medicines_marker):raw_advice.find(self_care_marker)].strip())
    if self_care_marker in raw_advice:
        relevant_advice.append(raw_advice[raw_advice.find(self_care_marker):].strip())

    # Combine the sections into a single advice string
    return " ".join(relevant_advice)

# Text-to-Speech (TTS) 

def generate_tts_with_gtts(text):
    """
    Convert the given health advice text into speech using the gTTS.
    The speech is saved as an audio file.
    """
    try:
        if text.strip():
            tts = gTTS(text)
            output_file = "output_advice.mp3"
            tts.save(output_file)
            print(f"Audio saved as '{output_file}'.")
            os.system(f"mpg123 {output_file}")  # Plays the audio file
        else:
            print("No text to convert to speech.")
    except Exception as e:
        print(f"Error in TTS generation with gTTS: {e}")

# ----------------- Speech Recognition ----------------- #

def transcribe_audio(audio_path):
    """
    Convert speech from an audio file into text using Google Speech Recognition.
    This function processes the user's voice input from a recorded file.
    """
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)
    try:
        return recognizer.recognize_google(audio)
    except sr.UnknownValueError:
        return "Could not understand the audio."
    except sr.RequestError as e:
        return f"Speech recognition service error: {e}"

# ----------------- Main Program ----------------- #

if __name__ == "__main__":
    # Load the symptoms and diseases dataset
    file_path = "Final_Augmented_dataset_Diseases_and_Symptoms.csv"
    symptom_data = load_symptom_data(file_path)

    # Step 1: Transcribe the user input (audio file)
    audio_file = "recording.wav"
    spoken_text = transcribe_audio(audio_file)
    print(f"Recognized Text: {spoken_text}")

    # Step 2: Process the transcribed text to extract symptoms
    symptoms_data = process_user_input(spoken_text)
    print(f"Processed Symptoms Data: {symptoms_data}")

    # Step 3: Analyze the symptoms and identify possible conditions
    condition_result, conditions = analyze_symptoms(symptoms_data, symptom_data)
    print(condition_result)

    # Step 4: Generate health advice based on identified symptoms and conditions
    raw_advice = generate_health_advice(symptoms_data, conditions)
    print(f"Generated Advice:\n{raw_advice}")

    # Step 5: Extract relevant advice sections and generate TTS output
    relevant_advice = extract_relevant_advice(raw_advice)
    print(f"Relevant Advice for TTS:\n{relevant_advice}")
    generate_tts_with_gtts(relevant_advice)

    print("Program finished successfully.")
